# Theory scaling validation — γ-fit + Yeom overlay

Validates two theoretical anchors:

  1. Closed-set re-ID top-1 as a function of cohort size N — fits the exponent γ in `top1 ≈ 1 - C * N^(1 - γ)` for both EEGNet and Riemann tangent-space.
  2. DP-SGD empirical re-ID top-1 vs the Yeom 2018 MI-advantage upper bound `1 - exp(-ε) - δ`, using whatever `results/29_d3_eps_sweep.json` is available in the repo.

**Runtime → L4 GPU.** Expected wall: ~50-70 min (6 cohort sizes × EEGNet train + 6 cohort sizes × Riemann fit + closed-set attacks).

**Don't `Save a copy in GitHub` from Colab.**

In [ ]:
!rm -rf /content/bci-identity-leakage
!git clone https://github.com/manrajmondair/bci-identity-leakage.git /content/bci-identity-leakage
%cd /content/bci-identity-leakage
!git rev-parse HEAD
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())

In [ ]:
import torch
tv = torch.__version__.split('+')[0]
!pip install --quiet "torchaudio=={tv}"
!pip install --quiet mne moabb pyriemann braindecode opacus httpx tqdm rich scipy

In [ ]:
!python -m data.prefetch_direct --runs imagery --workers 8

In [ ]:
!PYTHONUNBUFFERED=1 python -u -m experiments.30_theory_scaling --all

In [ ]:
import json, os, subprocess, datetime, platform, sys
import torch
PROJECT_DIR = '/content/bci-identity-leakage'
if os.path.isdir(PROJECT_DIR): os.chdir(PROJECT_DIR)
RESULTS = 'results/30_theory_scaling.json'
TAG = 'theory_scaling'
try:
    git_sha = subprocess.check_output(['git','rev-parse','HEAD']).decode().strip()
except Exception: git_sha = 'unknown'
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
now_utc = datetime.datetime.utcnow().replace(microsecond=0).isoformat() + 'Z'
run_id = now_utc.replace(':','').replace('-','').rstrip('Z') + f'_{TAG}_{git_sha[:7]}'
meta = {'run_id': run_id, 'experiment': 'experiments.30_theory_scaling',
        'git_sha': git_sha, 'hardware': f'Colab {gpu}',
        'platform': platform.platform(), 'torch_version': torch.__version__,
        'completed_at_utc': now_utc, 'outputs': [RESULTS]}
os.makedirs(f'runs/{run_id}', exist_ok=True)
with open(f'runs/{run_id}/meta.json','w') as f: json.dump(meta, f, indent=2)
if not os.path.exists(RESULTS):
    sys.exit(f'!!! {RESULTS} missing.')
print('--- BEGIN 30_theory_scaling.json ---')
print(open(RESULTS).read())
print('--- END 30_theory_scaling.json ---')
print()
print('--- BEGIN run meta ---')
print(json.dumps(meta, indent=2))
print('--- END run meta ---')